# Mesure de transmission de la lentille

https://rubinobs.atlassian.net/wiki/spaces/LTS/pages/50086854/CBP+Calibration+System+Hardware#Lens


In [ ]:
import os 
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.pyplot import cm
import pickle
from astropy.constants import e, h, c
from scipy.interpolate import interp1d
import pandas as pd
from tcbp.reduction.dataset import get_calib_path

from tcbp.analysis.lsst import LSSTRun

%matplotlib widget

In [ ]:
def get_data(run, doplot=True):
    """
    Compute and plot charges ratio between sphere and receiver photodiodes for one run

    run : LSSTRun
        LSSTRun object containing the data for the run
    doplot : bool
        If True, the results are plotted
    """

    ### Get data
    wls = run["set_wl"]
    
    Q_sphere = run['pd_charge_per_burst'][wls > 0.]
    Q_receiver = run['calib_charge_per_burst'][wls > 0.]

    Q_sphere_err = run['pd_charge_per_burst_err'][wls > 0.]
    Q_receiver_err = run['calib_charge_per_burst_err'][wls > 0.]

    ratio = np.abs(Q_receiver / Q_sphere)
    ratio_err = np.abs(ratio) * np.sqrt((Q_sphere_err / Q_sphere)**2 + (Q_receiver_err / Q_receiver)**2)

    ### Plot
    if doplot:
        fig, (ax1,ax2,ax3) = plt.subplots(3, sharex="all", figsize=(8,8))

        ax1.errorbar(wls[wls > 0.], Q_sphere, yerr=Q_sphere_err, c='b', marker='.', ls="")
        ax1.set_ylabel('Charges Sphere [C]')
            
        ax2.errorbar(wls[wls > 0.], Q_receiver, yerr=Q_receiver_err, c='r', marker='.', ls="")
        ax2.set_ylabel('Charges Receiver [C]')

        ax3.errorbar(wls[wls > 0.], ratio, yerr=ratio_err, c='indigo', marker='.', ls="")
        ax3.set_ylabel('Ratio Receiver / Sphere [No unit]')
        ax3.set_xlabel('wavelengths [nm]')

        fig.tight_layout()
        plt.show()

    return ratio, ratio_err

## Data file

In [ ]:
main_repo = '/home/lmousset/Projets_Recherche/LSST/tCBP_optical_bench/data/'

### Avec la cellule solaire
#data_dir = main_repo + '20260701_optical_bench_lens_transmission/'  # Tout premier run, très bruité à cause des long temps de pause
#data_dir = main_repo + '20260701_171854_optical_bench_lens_transmission/' # Avec lentille, scan complet, petits tps de pause x 5
#data_dir = main_repo + '20260701_184358_optical_bench_lens_transmission/' # Avec lentille, de 300 à 400nm seulement, petits tps de pause x 5
#data_dir = main_repo + '20260701_193647_optical_bench_lens_transmission/' # Sans lentille, scan complet, petits tps de pause x 5

### Avec la mini lentille
#data_dir = main_repo + '20260703_145059_optical_bench_lens_transmission_pinhole5mm_MiniLensNoBig_slit20/' # Sans grosse lentille
#data_dir = main_repo + '20260703_163504_optical_bench_lens_transmission_pinhole5mm_MiniLensWithBig_slit20/' # Avec grosse lentille
#data_dir = main_repo + '20260703_190329_optical_bench_lens_transmission_pinhole5mm_MiniLensWithBig_slit20/' # Avec grosse lentille  Le montage a bougé entre les deux runs, donc pas comparable
#data_dir = main_repo + '20260703_200311_optical_bench_lens_transmission_pinhole5mm_MiniLensNoBig_slit20/' # Sans grosse lentille
#data_dir = main_repo + '20260706_140844_optical_bench_lens_transmission_pinhole5mm_MiniLensWithBig_slit20/' # Avec grosse lentille
#data_dir = main_repo + '20260706_185224_optical_bench_lens_transmission_pinhole5mm_MiniLensNoBig_slit20/' # Sans grosse lentille
#data_dir = main_repo + '20260706_195018_optical_bench_lens_transmission_pinhole5mm_MiniLensBigLensReversed_slit20/' # Avec grosse lentille retournée
#data_dir = main_repo + '20260707_174705_optical_bench_lens_transmission_pinhole5mmIris25_MiniLensWithBigOpenIris_slit20/' # Avec grosse lentille + iris ouvert 
#data_dir = main_repo + '20260707_193623_optical_bench_lens_transmission_pinhole5mmIris25_MiniLensWithBigCloseIris_slit20/' # Avec grosse lentille + iris fermé
data_dir = main_repo + '20260707_203859_optical_bench_lens_transmission_pinhole5mmIris25_MiniLensNoBigCloseIris_slit20/' # Sans grosse lentille + iris fermé
#data_dir = main_repo + '20260707_213549_optical_bench_lens_transmission_pinhole5mmIris25_MiniLensNoBigOpenIris_slit20/' # Sans grosse lentille + iris ouvert

### En projetant l'image de la mini lens par la grosse lentille
#data_dir = main_repo + '20260706_154719_optical_bench_lens_transmission_pinhole5mm_MiniLensWithBigLongProj_slit20/' # Avec grosse lentille
#data_dir = main_repo + '20260706_170942_optical_bench_lens_transmission_pinhole5mm_MiniLensWithBigLongProjMask_slit20/' # Avec grosse lentille + mask


### Image d'un pinhole de 1mm sur photodiode (pas de mini lens) => Très bruité
#data_dir = main_repo + '20260706_215451_optical_bench_lens_transmission_pinhole1mm_BigLensLongProj_slit20/' # Avec grosse lentille
#data_dir = main_repo + '20260706_234151_optical_bench_lens_transmission_pinhole1mm_NoLensShortProj_slit20/' # Sans grosse lentille, photodiode à la sortie du tube


### Avec le doublet achromatique
#data_dir = main_repo + '20260707_134736_optical_bench_lens_transmission_pinhole1mm_DoubletAcNoBigLens_slit20/' # Avec doublet achromatique, sans grosse lentille
#data_dir = main_repo + '20260707_152515_optical_bench_lens_transmission_pinhole1mm_DoubletAcWithBigLens_slit20/' # Avec doublet achromatique, avec grosse lentille

## Reduce the data

Here we build the run.py file.

In [ ]:
pd_extname="KEITHLEY" # Sphere
sc_extname="KEYSIGHT" # NIST

## Do the reduction
run = LSSTRun(data_dir,
              nbursts=1, spectro_calib_path="2026_Auxtel/20260429", 
              spectro_temp_calib=False,
              pd_extname=pd_extname, sc_extname=sc_extname,
              pins={"KEYSIGHT":1, "KEITHLEY":4, "SHUTTER":2},
              ticks_per_shutter_event=1,
              use_digital_analyzer=True)
run.load()
run.extraction()
run.plot_summary()

np.save(data_dir + "run.npy", run.data)

In [ ]:
### Load a run file
run = np.load(data_dir + "run.npy", allow_pickle=True)
wls_run = run["set_wl"][run["set_wl"] > 300.]
ratio, ratio_err = get_data(run, doplot=True)

print(np.mean(np.abs(ratio_err) / ratio))

## Lens transmission measured by Elana

In [ ]:
data_dir2 = '/home/lmousset/Projets_Recherche/LSST/Mesure_transmission_tcbp/'

with open(data_dir2 + 'Interp_lens_transmission.pkl', 'rb') as file:
    interpolator_lens = pickle.load(file)

transmission_lens_interp = interpolator_lens(wls_run)


csv_file = data_dir2 + 'lens_ratio_calibration_system_2.csv'

df_lens = pd.read_csv(csv_file)
print(df_lens.head())

wls_lens = df_lens['Wavelength'].values
transmission_lens = df_lens['lens_ratio'].values
lens_ratio_error = df_lens['ratio_error'].values

plt.figure(figsize=(10,5))
plt.plot(wls_run, transmission_lens_interp, c='green', marker='.', ls="")
plt.plot(wls_lens, transmission_lens, c='orange', marker='.', ls="")
plt.ylim(0, 1)


## Get several runs

In [ ]:
### Cellule solaire
#data_dir_list = [main_repo + '20260701_optical_bench_lens_transmission/',        # Premier run, très bruité à cause des long temps de pause
                 #main_repo + '20260701_171854_optical_bench_lens_transmission/', # Avec lentille 
                 #main_repo + '20260701_184358_optical_bench_lens_transmission/', # Avec lentille, de 300 à 400nm seulement
                 #main_repo + '20260701_193647_optical_bench_lens_transmission/'  # Sans lentille
                 #]
### Mini lentille
data_dir_list = [main_repo + '20260703_145059_optical_bench_lens_transmission_pinhole5mm_MiniLensNoBig_slit20/', # Sans grosse lentille
                 main_repo + '20260703_163504_optical_bench_lens_transmission_pinhole5mm_MiniLensWithBig_slit20/',  # Avec grosse lentille
                 #main_repo + '20260703_190329_optical_bench_lens_transmission_pinhole5mm_MiniLensWithBig_slit20/',  # Avec grosse lentille Le montage a bougé entre les deux runs, donc pas comparable
                 #main_repo + '20260703_200311_optical_bench_lens_transmission_pinhole5mm_MiniLensNoBig_slit20/',  # Sans grosse lentille
                 #main_repo + '20260706_140844_optical_bench_lens_transmission_pinhole5mm_MiniLensWithBig_slit20/',  # Avec grosse lentille
                 #main_repo + '20260706_154719_optical_bench_lens_transmission_pinhole5mm_MiniLensWithBigLongProj_slit20/',  # Avec grosse lentille
                 #main_repo + '20260706_170942_optical_bench_lens_transmission_pinhole5mm_MiniLensWithBigLongProjMask_slit20/',  # Avec grosse lentille + mask
                 main_repo + '20260706_185224_optical_bench_lens_transmission_pinhole5mm_MiniLensNoBig_slit20/',  # Sans grosse lentille
                 main_repo + '20260706_195018_optical_bench_lens_transmission_pinhole5mm_MiniLensBigLensReversed_slit20/',  # Avec grosse lentille retournée
                 #main_repo + '20260706_215451_optical_bench_lens_transmission_pinhole1mm_BigLensLongProj_slit20/',  # Avec grosse lentille, image d'un pinhole de 1mm sur photodiode (pas de mini lens)
                 #main_repo + '20260706_234151_optical_bench_lens_transmission_pinhole1mm_NoLensShortProj_slit20/',  # Sans grosse lentille, photodiode à la sortie du tube
                 main_repo + '20260707_134736_optical_bench_lens_transmission_pinhole1mm_DoubletAcNoBigLens_slit20/',  # Avec doublet achromatique, sans grosse lentille
                 main_repo + '20260707_152515_optical_bench_lens_transmission_pinhole1mm_DoubletAcWithBigLens_slit20/',  # Avec doublet achromatique, avec grosse lentille
                 main_repo + '20260707_174705_optical_bench_lens_transmission_pinhole5mmIris25_MiniLensWithBigOpenIris_slit20/',  # Avec grosse lentille + iris ouvert
                 main_repo + '20260707_193623_optical_bench_lens_transmission_pinhole5mmIris25_MiniLensWithBigCloseIris_slit20/',  # Avec grosse lentille + iris fermé
                 main_repo + '20260707_203859_optical_bench_lens_transmission_pinhole5mmIris25_MiniLensNoBigCloseIris_slit20',  # Sans grosse lentille + iris fermé
                 main_repo + '20260707_213549_optical_bench_lens_transmission_pinhole5mmIris25_MiniLensNoBigOpenIris_slit20'  # Sans grosse lentille + iris ouvert
                 ]

In [ ]:
data_dict = {'20260701_171854': ['Solar cell', 'Avec lentille'],
             '20260701_184358': ['Solar cell', 'Avec lentille, de 300 à 400nm seulement'],
             '20260701_193647': ['Solar cell', 'Sans lentille'],
             '20260703_145059': ['Mini lens', 'Sans grosse lentille'],
             '20260703_163504': ['Mini lens', 'Avec grosse lentille'],
             '20260703_190329': ['Mini lens', 'Avec grosse lentille  Le montage a bougé entre les deux runs, donc pas comparable'],
             '20260703_200311': ['Mini lens', 'Sans grosse lentille'],
             '20260706_140844': ['Mini lens', 'Avec grosse lentille'],
             '20260706_154719': ['Mini lens', 'Avec grosse lentille'],
             '20260706_170942': ['Mini lens', 'Avec grosse lentille + mask'],
             '20260706_185224': ['Mini lens', 'Sans grosse lentille'],
             '20260706_195018': ['Mini lens', 'Avec grosse lentille retournée'],
             '20260706_215451': ['Mini lens', 'Avec grosse lentille'],
             '20260706_234151': ['Mini lens', 'Sans grosse lentille, photodiode à la sortie du tube'],
             '20260707_134736': ['Doublet achromatique', 'Avec doublet achromatique, sans grosse lentille'],
             '20260707_152515': ['Doublet achromatique', 'Avec doublet achromatique, avec grosse lentille'],
             '20260707_174705': ['Mini lens', 'Avec grosse lentille + iris ouvert'],
             '20260707_193623': ['Mini lens', 'Avec grosse lentille + iris fermé'],
             '20260707_203859': ['Mini lens', 'Sans grosse lentille + iris fermé'],
             '20260707_213549': ['Mini lens', 'Sans grosse lentille + iris ouvert']
             }

In [ ]:
nruns = len(data_dict.keys())

all_ratio = []
all_ratio_err = []
all_wls_run = []
for i, key in enumerate(data_dict.keys()):
    print(f'Run {i}: {key}')
    matching_dirs = [d for d in os.listdir(main_repo) if d.startswith(key)]

    data_dir = os.path.join(main_repo, matching_dirs[0])
    print(data_dir)

    run = np.load(data_dir + "/run.npy", allow_pickle=True)

    wls_run = run["set_wl"][run["set_wl"] > 0.]

    #print(wls_run)
    ratio, ratio_err = get_data(run, doplot=False)

    all_wls_run.append(wls_run)
    all_ratio.append(ratio)
    all_ratio_err.append(ratio_err)

    print(np.mean(np.abs(ratio_err) / ratio))

In [ ]:
nruns = len(data_dir_list)

all_ratio = []
all_ratio_err = []
all_wls_run = []
for i, data_dir in enumerate(data_dir_list):
    print(i)

    run = np.load(data_dir + "run.npy", allow_pickle=True)

    wls_run = run["set_wl"][run["set_wl"] > 0.]

    print(wls_run)
    ratio, ratio_err = plot_Qphotodiodes(run)


    all_wls_run.append(wls_run)
    all_ratio.append(ratio)
    all_ratio_err.append(ratio_err)

    print(np.mean(np.abs(ratio_err) / ratio))

In [ ]:
# Sort by wavelength
all_wls_run_sorted = []
for i in range(nruns):
    sort_indices = np.argsort(all_wls_run[i])
    all_wls_run_sorted.append(all_wls_run[i][sort_indices])
    all_ratio[i] = all_ratio[i][sort_indices]
    all_ratio_err[i] = all_ratio_err[i][sort_indices]

In [ ]:
## Plot all ratios
plt.figure(figsize=(10,5))
for i in range(nruns):
    print(i)
    #norm = all_ratio[i][all_wls_run_sorted[i]==769.996][0]
    #norm = all_ratio[i][all_wls_run_sorted[i]==500.][0]
    
    plt.errorbar(all_wls_run_sorted[i], all_ratio[i], yerr=np.abs(all_ratio_err[i]), marker='.', ls="", label=f'Run {i}')
plt.xlabel('wavelengths [nm]')
plt.ylabel('ratio Receiver/Sphere')
plt.grid(True)
#plt.ylim(0, 2.5)
plt.legend()

In [ ]:
all_ratio_avg = []
all_ratio_err_avg = []
all_wls_avg = []
for i in range(nruns):
    print("Run", i)
    if (all_wls_run_sorted[i] == np.unique(all_wls_run_sorted[i])).all():
        print("Wavelengths are unique, no need to average")
        all_wls_avg.append(all_wls_run_sorted[i])
        all_ratio_avg.append(all_ratio[i])
        all_ratio_err_avg.append(all_ratio_err[i])
    else:
        wls_set = np.unique(all_wls_run_sorted[i])
        avg = [np.mean(all_ratio[i][all_wls_run_sorted[i]==k]) for k in wls_set]
        err_avg = [np.std(all_ratio_err[i][all_wls_run_sorted[i]==k])/np.sqrt(len(all_ratio_err[i][all_wls_run_sorted[i]==k])) for k in wls_set]

        all_wls_avg.append(wls_set)
        all_ratio_avg.append(np.array(avg))
        all_ratio_err_avg.append(np.array(err_avg))

print(all_wls_avg[1].shape, all_ratio_avg[1].shape)

In [ ]:
## Plot all ratios
plt.figure(figsize=(10,5))
for i in range(nruns):
    print(i)    
    plt.errorbar(all_wls_avg[i], all_ratio_avg[i], yerr=np.abs(all_ratio_err_avg[i]), marker='.', ls="", label=f'Run {i}')
plt.xlabel('wavelengths [nm]')
plt.ylabel('ratio Receiver/Sphere')
plt.grid(True)
#plt.ylim(0, 2.5)
plt.legend()

## Lens Transmission

In [ ]:
## Rapport entre 2 scans
all_commun_wls = []
all_transmissions = []
all_transmission_errs = []
plt.figure(figsize=(10,5))
for i, (irun1, irun2) in enumerate([(1, 0),(3, 2), (7, 8), (6, 9)]):

    commun_wls = np.intersect1d(all_wls_avg[irun1], all_wls_avg[irun2])
    print(f"Commun wavelengths between run {irun1} and run {irun2}: {commun_wls}")

    # Utiliser searchsorted pour maintenir l'ordre correct des longueurs d'onde
    indices1 = np.searchsorted(all_wls_avg[irun1], commun_wls)
    indices2 = np.searchsorted(all_wls_avg[irun2], commun_wls)

    commun_ratio1 = all_ratio_avg[irun1][indices1]
    commun_ratio2 = all_ratio_avg[irun2][indices2]

    commun_ratio_err1 = all_ratio_err_avg[irun1][indices1]
    commun_ratio_err2 = all_ratio_err_avg[irun2][indices2]

    rapport = commun_ratio1 / commun_ratio2
    rapport_err = np.sqrt((commun_ratio_err1 / commun_ratio1)**2 + (commun_ratio_err2 / commun_ratio2)**2) * rapport
    print(rapport_err)

    #if i in [2]:
     #   rapport *= 1.6
      #  rapport_err *= 1.6
    
    all_commun_wls.append(commun_wls)
    all_transmissions.append(rapport)
    all_transmission_errs.append(rapport_err)

    plt.errorbar(commun_wls, rapport, yerr=rapport_err, marker='.', ls="", label=f'Run {irun1} / Run {irun2}')

plt.errorbar(wls_lens, transmission_lens, yerr=lens_ratio_error, c='k', marker='.', ls="--",  label='Elana measurement')
#plt.ylim(0.99, 1.01)
plt.ylabel('Lens transmission')
plt.xlabel('wavelengths [nm]')
plt.legend()

In [ ]:
## Plot le rapport entre 2 transmissions

plt.figure(figsize=(10,5))
for i, (t1, t2) in enumerate([(2, 3)]):

    commun_wls = np.intersect1d(all_commun_wls[t1], all_commun_wls[t2])
    print(f"Commun wavelengths between run {t1} and run {t2}: {commun_wls}")

    # Utiliser searchsorted pour maintenir l'ordre correct des longueurs d'onde
    indices1 = np.searchsorted(all_commun_wls[t1], commun_wls)
    indices2 = np.searchsorted(all_commun_wls[t2], commun_wls)

    commun_ratio1 = all_transmissions[t1][indices1]
    commun_ratio2 = all_transmissions[t2][indices2]

    commun_ratio_err1 = all_transmission_errs[t1][indices1]
    commun_ratio_err2 = all_transmission_errs[t2][indices2]

    rapport = commun_ratio1 / commun_ratio2
    rapport_err = np.sqrt((commun_ratio_err1 / commun_ratio1)**2 + (commun_ratio_err2 / commun_ratio2)**2) * rapport
    print(rapport_err)

    plt.errorbar(commun_wls, rapport, yerr=rapport_err, marker='.', ls="", label=f'Transmission {t1} / Transmission {t2}')

#plt.ylim(0.99, 1.01)
plt.ylabel('Ratio between 2 transmissions')
plt.xlabel('wavelengths [nm]')
plt.legend()

In [ ]:
# Save the transmission lens data
transmission_lens_to_save = np.zeros((3, len(commun_wls)))
transmission_lens_to_save[0, :] = commun_wls
transmission_lens_to_save[1, :] = rapport
transmission_lens_to_save[2, :] = rapport_err

np.save("/home/lmousset/Projets_Recherche/LSST/tCBP_optical_bench/lens_transmission_data_new.npy", transmission_lens_to_save)